In [1]:
%pip install sentence_transformers

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
%pip install chromadb

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# Environment Setup and Imports for RAG and Embeddings
import os
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
import warnings
warnings.filterwarnings('ignore')

c:\Users\Administrator\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:

# Initialize sentence transformer for embeddings
print("\n🧠 Loading Embedding Model...")
try:
    embedding_model = SentenceTransformer('all-MiniLM-L6-v2') 
    print("✅ SentenceTransformer model loaded successfully")
except Exception as e:
    print(f"⚠️ Could not load SentenceTransformer: {e}")



🧠 Loading Embedding Model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 219.70it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ SentenceTransformer model loaded successfully


In [5]:
# 🔬 Exploring SentenceTransformer Capabilities
print("🧠 TESTING SENTENCETRANSFORMER MODEL")
print("="*60)

# Test 1: Basic text encoding
print("\n1️⃣ BASIC TEXT ENCODING:")
print("-"*30)
test_sentences = [
    "The cat is sleeping on the couch",
    "A feline is resting on the sofa", 
    "The dog is barking loudly",
    "Python is a programming language",
    "Machine learning is fascinating"
]

# Create embeddings for test sentences
embeddings = embedding_model.encode(test_sentences)
# Takes collection of text sentences (stored in test_sentences) and transforms them into dense numerical vectors that capture their semantic meaning 
# through a series of steps including  tokenization of the text, passing it through transformer layers, and applying pooling strategies to create
# fixed-size representations regardless of input length.
# 5*384 = embeddings.shape
print(f"✅ Created embeddings for {len(test_sentences)} sentences")
print(f"📊 Each embedding has {embeddings.shape[1]} dimensions")
print(f"🔢 Embedding shape: {embeddings.shape}")
print(type(embeddings))

🧠 TESTING SENTENCETRANSFORMER MODEL

1️⃣ BASIC TEXT ENCODING:
------------------------------
✅ Created embeddings for 5 sentences
📊 Each embedding has 384 dimensions
🔢 Embedding shape: (5, 384)
<class 'numpy.ndarray'>


In [6]:
# Test 2: Semantic similarity demonstration
print("\n2️⃣ SEMANTIC SIMILARITY DEMO:")
print("-"*35)
sentence1 = "The cat is sleeping on the couch"
sentence2 = "A feline is resting on the sofa"
sentence3 = "The dog is barking loudly"
# Get embeddings
emb1 = embedding_model.encode([sentence1])
emb2 = embedding_model.encode([sentence2])
emb3 = embedding_model.encode([sentence3])
# Calculate similarities
similarity_1_2 = cosine_similarity(emb1, emb2)[0][0]
similarity_1_3 = cosine_similarity(emb1, emb3)[0][0]
# cosine_similarity() always returns a 2D matrix, even when comparing just two single vectors
# The first [0] extracts the first (and only) row from this matrix, yielding a 1D array, and the second
#  [0] extracts the first (and only)# element from that array, giving you the actual 
# similarity score as a scalar value.
print(f"📝 Sentence 1: '{sentence1}'")
print(f"📝 Sentence 2: '{sentence2}'")
print(f"📝 Sentence 3: '{sentence3}'")
print()
print(f"🔍 Similarity between 1 & 2 (similar meaning): {similarity_1_2:.3f}")
print(f"🔍 Similarity between 1 & 3 (different meaning): {similarity_1_3:.3f}")


2️⃣ SEMANTIC SIMILARITY DEMO:
-----------------------------------
📝 Sentence 1: 'The cat is sleeping on the couch'
📝 Sentence 2: 'A feline is resting on the sofa'
📝 Sentence 3: 'The dog is barking loudly'

🔍 Similarity between 1 & 2 (similar meaning): 0.557
🔍 Similarity between 1 & 3 (different meaning): 0.166


In [7]:

# Test 3: Model information
print("\n3️⃣ MODEL INFORMATION:")
print("-"*25)
print(f"📋 Model name: all-MiniLM-L6-v2")
print(f"🏢 Provider: sentence-transformers")
print(f"📏 Max sequence length: {embedding_model.max_seq_length}")
# reveals the maximum number of tokens that the model can process in a single input. For the 'all-MiniLM-L6-v2' model, this is typically 256 tokens. 
print(f"🧮 Embedding dimension: {embedding_model.get_sentence_embedding_dimension()}")
# displays the dimensionality of the output vectors produced by the model. For 'all-MiniLM-L6-v2', this returns 384 dimensions, which represents the length of each embedding vector.



3️⃣ MODEL INFORMATION:
-------------------------
📋 Model name: all-MiniLM-L6-v2
🏢 Provider: sentence-transformers
📏 Max sequence length: 256
🧮 Embedding dimension: 384


In [8]:
# Test 4: What makes this powerful for RAG
print("\n4️⃣ WHY THIS IS POWERFUL FOR RAG:")
print("-"*40)
rag_examples = [
    "How do I reset my password?",
    "Password reset instructions",
    "What is machine learning?",
    "Define artificial intelligence",
    "How to deploy a Flask app?"
]
rag_embeddings = embedding_model.encode(rag_examples)
query = "I forgot my password, how can I reset it?"
query_embedding = embedding_model.encode([query])
print(rag_embeddings[0][:5], "...")  # Print first 5 dimensions  of the  embedding for the first RAG example
print(rag_embeddings[1][:5], "...")  # Print first 5 dimensions of the  embedding for the second RAG example
print(rag_embeddings[2][:5], "...")  
print(rag_embeddings[3][:5], "...")
print(rag_embeddings[4][:5], "...")
print()
print(query_embedding[0][:5], "...")  # Print first 5 dimensions of the query embedding


4️⃣ WHY THIS IS POWERFUL FOR RAG:
----------------------------------------
[ 0.00690308 -0.0630234  -0.07204397 -0.03249606 -0.04270937] ...
[-0.05100249 -0.03911759 -0.02330978 -0.01806637 -0.05207053] ...
[-0.01995453  0.00987804  0.01024961  0.02955366  0.02718642] ...
[-0.02191449 -0.01889791  0.02544149  0.03250609 -0.00174214] ...
[-0.00445291 -0.0066343  -0.07668515 -0.04697259  0.06898064] ...

[-0.02587559 -0.04078115 -0.06764195  0.00999548 -0.03064711] ...


In [9]:
similarities = cosine_similarity(query_embedding, rag_embeddings)[0]
print(similarities) # Print similarity scores for each document


[0.936839   0.74566996 0.13208576 0.10546421 0.06133908]


In [10]:

best_match_idx = np.argmax(similarities) # Finds the index of the highest similarity score, ie query's best match in the knowledge base
print(f"\n✅ Best match found: '{rag_examples[best_match_idx]}'")


✅ Best match found: 'How do I reset my password?'


In [11]:
best_match_idx= np.argmax(similarities) # Finds the index of the highest similarity score, ie query's best match in the knowledge base
print(f"🔍 User Query: '{query}'")
print(f"📚 Knowledge Base:")
for i, doc in enumerate(rag_examples):
    similarity_score = similarities[i]
    marker = "🎯" if i == best_match_idx else "📄"
    print(f"   {marker} '{doc}' (similarity: {similarity_score:.3f})")

    

🔍 User Query: 'I forgot my password, how can I reset it?'
📚 Knowledge Base:
   🎯 'How do I reset my password?' (similarity: 0.937)
   📄 'Password reset instructions' (similarity: 0.746)
   📄 'What is machine learning?' (similarity: 0.132)
   📄 'Define artificial intelligence' (similarity: 0.105)
   📄 'How to deploy a Flask app?' (similarity: 0.061)


In [12]:
rag_examples = [
    "How do I reset my password?",
    "Password reset instructions",
    "What is machine learning?",
    "Define artificial intelligence",
    "How to deploy a Flask app?"
]
rag_embeddings = embedding_model.encode(rag_examples)

query = ["I forgot my password, how can I reset it?", "Explain machine learning concepts." ]
query_embedding = embedding_model.encode(query)

print(rag_embeddings[0][:5], "...")  # Print first 5 dimensions of the  embedding for the first RAG example
print(rag_embeddings[1][:5], "...")  # Print first 5 dimensions of the  embedding for the second RAG example
print(rag_embeddings[2][:5], "...")
print(rag_embeddings[3][:5], "...")
print(rag_embeddings[4][:5], "...")
print()
print(query_embedding[0][:5], "...")  # Print first 5 dimensions of the query embedding
print(query_embedding[1][:5], "...")  # Print first 5 dimensions of the second query embedding



[ 0.00690308 -0.0630234  -0.07204397 -0.03249606 -0.04270937] ...
[-0.05100249 -0.03911759 -0.02330978 -0.01806637 -0.05207053] ...
[-0.01995453  0.00987804  0.01024961  0.02955366  0.02718642] ...
[-0.02191449 -0.01889791  0.02544149  0.03250609 -0.00174214] ...
[-0.00445291 -0.0066343  -0.07668515 -0.04697259  0.06898064] ...

[-0.02587559 -0.04078115 -0.06764195  0.00999548 -0.03064711] ...
[-0.000408   -0.03420475  0.00674587 -0.01755425  0.03677078] ...


In [13]:
similarities = cosine_similarity(query_embedding, rag_embeddings)
print(similarities) # Print similarity scores for each document


[[ 0.9368392   0.74567     0.13208576  0.10546421  0.06133908]
 [ 0.09863128  0.13525428  0.77126217  0.39019448 -0.07563737]]


In [14]:
best_match_idx= np.argmax(similarities[0]) # Finds the index of the highest similarity score for the first query, ie best match in the knowledge base
print(f"\n✅ Best match found for query: '{query[0]}': '{rag_examples[best_match_idx]}'")


✅ Best match found for query: 'I forgot my password, how can I reset it?': 'How do I reset my password?'


In [15]:
best_match_idx= np.argmax(similarities[1]) # Finds the index of the highest similarity score for the second query, ie best match in the knowledge base
print(f"\n✅ Best match found for query: '{query[1]}': '{rag_examples[best_match_idx]}'")


✅ Best match found for query: 'Explain machine learning concepts.': 'What is machine learning?'


In [18]:
import chromadb
from sentence_transformers import SentenceTransformer

# Create a client
chroma_client = chromadb.Client()

# Create a collection 
collection1 = chroma_client.create_collection(
    name="my_collection",
    get_or_create=True
)
print(type(collection1))
# Add documents with real embeddings generated by sentence-transformers
documents = [
    "This is a document about pineapple",
    "This is a document about oranges",
    "This is vector db example"
]

# Generate embeddings for all documents
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
doc_embeddings = embedding_model.encode(documents).tolist()

collection1.add(
    ids=["id1", "id2", "id3"],
    documents=documents,
    embeddings=doc_embeddings,
    metadatas=[  #manually added metadata for each document
        {"source": "sample.pdf", "category": "fruits"},
        {"source": "sample.pdf", "category": "fruits"},
        {"source": "sample.pdf", "category": "ai"}
    ]
)

# Use sentence-transformers to generate an embedding for a query sentence
#embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
query_sentence = "Find documents about pineapple"
query_embedding = embedding_model.encode([query_sentence]).tolist()

results = collection1.query(
    query_embeddings=query_embedding, # Use generated embedding from sentence
    n_results=3
    #where={"category": "ai"} # Uncomment to filter by metadata
)
print(results)


# Get the list of distances and documents
distances = results['distances'][0]
documents = results['documents'][0]

# Find the index of the minimum distance
best_idx = distances.index(min(distances))

# Print the most similar document
print("Most similar document:", documents[best_idx])
print("Distance:", distances[best_idx])




<class 'chromadb.api.models.Collection.Collection'>


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 220.60it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'ids': [['id1', 'id2', 'id3']], 'embeddings': None, 'documents': [['This is a document about pineapple', 'This is a document about oranges', 'This is vector db example']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'source': 'sample.pdf', 'category': 'fruits'}, {'category': 'fruits', 'source': 'sample.pdf'}, {'source': 'sample.pdf', 'category': 'ai'}]], 'distances': [[0.3070813715457916, 1.113892912864685, 1.7240850925445557]]}
Most similar document: This is a document about pineapple
Distance: 0.3070813715457916


In [17]:
import chromadb
from sentence_transformers import SentenceTransformer

# Initialize embedding model (reuse throughout)
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# Create ChromaDB client and collection with cosine metric
chroma_client = chromadb.Client()
collection1 = chroma_client.create_collection(
    name="my_collection",
    metadata={"hnsw:space": "cosine"},
    get_or_create=True
)

# Add meaningful documents with metadata
documents = [
    "Pineapple is rich in bromelain and vitamin C, often discussed for digestive health.",
    "Oranges are citrus fruits high in vitamin C and fiber, commonly recommended for immune support.",
    "Vector databases store embeddings to enable fast semantic search over high-dimensional text data.",
    "Retrieval-Augmented Generation (RAG) retrieves relevant context to ground large language model answers.",
    "Azure Cosmos DB provides globally distributed, low-latency data access for scalable applications.",
    "MiniLM-based sentence embeddings capture semantic meaning in compact 384-dimensional vectors."
]

collection1.add(
    ids=[f"id{i+1}" for i in range(len(documents))],
    documents=documents,
    embeddings=embedding_model.encode(documents).tolist(),
    metadatas=[
        {"source": "nutrition_guide.pdf", "category": "fruits", "tags": "pineapple, nutrition"},
        {"source": "nutrition_guide.pdf", "category": "fruits", "tags": "orange, nutrition"},
        {"source": "vector_search_blog.md", "category": "ai", "tags": "vector-db, search"},
        {"source": "rag_whitepaper.pdf", "category": "ai", "tags": "rag, llm"},
        {"source": "azure_architecture.md", "category": "cloud", "tags": "azure, cosmosdb"},
        {"source": "embeddings_notes.md", "category": "ai", "tags": "embeddings, minilm"}
    ]
)

# Helper function to execute and display query results
def query_and_display(query_text, where_filter=None, n=3):
    print(f"\n{'='*60}")
    print(f"Query: '{query_text}'")
    if where_filter:
        print(f"Filter: {where_filter}")
    print('='*60)
    
    results = collection1.query(
        query_embeddings=embedding_model.encode([query_text]).tolist(),
        n_results=n,
        where=where_filter
    )
    
    # Unpack results
    distances = results['distances'][0]
    docs = results['documents'][0]
    metas = results.get('metadatas', [[{}]])[0]
    similarities = [1 - d for d in distances]
    
    # Display ranked results
    print("\nTop results (cosine similarity):")
    for rank, (doc, sim, meta) in enumerate(zip(docs, similarities, metas), start=1):
        print(f"{rank}. sim={sim:.3f} | {doc[:70]}... | {meta['category']}")
    
    return results

# Query 1: General search (all categories)
query_and_display("Which foods are good sources of vitamin C?")

# Query 2: Filtered search (AI category only)
query_and_display("How does RAG improve the answers from LLMs?", where_filter={"category": "ai"})


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 181.75it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Query: 'Which foods are good sources of vitamin C?'

Top results (cosine similarity):
1. sim=-0.544 | This is a document about pineapple... | fruits
2. sim=-0.684 | This is a document about oranges... | fruits
3. sim=-0.773 | Azure Cosmos DB provides globally distributed, low-latency data access... | cloud

Query: 'How does RAG improve the answers from LLMs?'
Filter: {'category': 'ai'}

Top results (cosine similarity):
1. sim=-0.050 | Retrieval-Augmented Generation (RAG) retrieves relevant context to gro... | ai
2. sim=-0.586 | MiniLM-based sentence embeddings capture semantic meaning in compact 3... | ai
3. sim=-0.827 | This is vector db example... | ai


{'ids': [['id4', 'id6', 'id3']],
 'embeddings': None,
 'documents': [['Retrieval-Augmented Generation (RAG) retrieves relevant context to ground large language model answers.',
   'MiniLM-based sentence embeddings capture semantic meaning in compact 384-dimensional vectors.',
   'This is vector db example']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'category': 'ai',
    'tags': 'rag, llm',
    'source': 'rag_whitepaper.pdf'},
   {'source': 'embeddings_notes.md',
    'tags': 'embeddings, minilm',
    'category': 'ai'},
   {'category': 'ai', 'source': 'sample.pdf'}]],
 'distances': [[1.049520492553711, 1.5860071182250977, 1.827392578125]]}